In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [2]:
df = pd.read_csv(r"C:\Users\OM\Downloads\train.txt",sep=';',header=None,names=['text','emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [5]:
df['emotion'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [6]:
emotion_numbers={}
i=0
for emo in df['emotion'].unique():
   emotion_numbers[emo]=i
   i+=1
df['emotion']=df['emotion'].map(emotion_numbers)    

In [7]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [8]:
df['text']=df['text'].apply(lambda x:x.lower())

In [9]:
import string
def remove_punc(txt):
   return txt.translate(str.maketrans('','',string.punctuation))  

In [10]:
df['text']=df['text'].apply(remove_punc)

In [11]:
def remove_numbers(txt):
    new=""
    for i in txt:
        if not i.isdigit():
            new+=i
    return new
df['text']=df['text'].apply(remove_numbers)

In [12]:
def emoji(txt):
   remain=''
   for i in txt:
     if i.isascii():
         remain+=i
   return remain
df['text']=df['text'].apply(emoji)

In [13]:
 import nltk

In [14]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [15]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\OM\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\OM\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
stop_words=set(stopwords.words('english'))

In [17]:
len(stop_words)

198

In [18]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [19]:
def remove_stopwords(txt):
  words = word_tokenize(txt)
  cleaned=[]
  for i in words:
      if not i in stop_words:
          cleaned.append(i)
  return ' '.join(cleaned)

In [20]:
df['text']=df['text'].apply(remove_stopwords)

In [21]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [22]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [23]:
from sklearn.model_selection import train_test_split

In [24]:
x=df['text']
y=df['emotion']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=42)

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

In [26]:
bow_vectorizer=CountVectorizer()
x_train_bow=bow_vectorizer.fit_transform(x_train)
x_test_bow=bow_vectorizer.transform(x_test)

In [39]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [40]:
nb_model=MultinomialNB()
nb_model.fit(x_train_bow,y_train)
y_pred_nb=nb_model.predict(x_test_bow)
print(accuracy_score(y_test,y_pred_nb))

0.7678125


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [42]:
tfidf_vectorizer=TfidfVectorizer()

x_train_tfidf=tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf=tfidf_vectorizer.transform(x_test)

In [43]:
nb_model2=MultinomialNB()
nb_model2.fit(x_train_tfidf,y_train)
y_pred_nb2=nb_model2.predict(x_test_tfidf)
print(accuracy_score(y_test,y_pred_nb2))

0.6609375


In [46]:
from sklearn.linear_model import LogisticRegression

In [50]:
lr_model=LogisticRegression(max_iter=1000)
lr_model.fit(x_train_bow,y_train)
y_pred_lr=lr_model.predict(x_test_bow)
print(accuracy_score(y_test,y_pred_lr))

0.88875


In [51]:
lr_model2=LogisticRegression(max_iter=1000)
lr_model2.fit(x_train_tfidf,y_train)
y_pred_lr2=lr_model2.predict(x_test_tfidf)
print(accuracy_score(y_test,y_pred_lr2))

0.8615625


In [52]:
print("Bag of Words + Logistic Regression given the highest validation/test accuracy, it's perfectly fine to choose it as your best model and the accuracy is:",accuracy_score(y_test,y_pred_lr))

Bag of Words + Logistic Regression given the highest validation/test accuracy, it's perfectly fine to choose it as your best model and the accuracy is: 0.88875
